# BERT
"BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding" (Devlin et al., 2018)

**Idea:** pretrain a Transformer **encoder** on unlabeled text by hiding random words and predicting them from **both** sides (masked language modeling), then reuse the same network for a downstream task by putting a small head on the `[CLS]` token and fine-tuning everything.

### BERT vs original Transformer vs GPT/SLM
| | Original Transformer (2017) | GPT / SLM | BERT |
|---|---|---|---|
| Parts | encoder + decoder | **decoder only** | **encoder only** |
| Attention mask | encoder: padding only; decoder: causal | causal (see only the past) | padding only -> **bidirectional** (every token sees left and right) |
| Training objective | supervised translation | next-token prediction | **masked LM** (+ next-sentence prediction in the paper) |
| Special tokens | `<bos>`, `<eos>` | `<bos>`/`<eos>` | `[CLS]` (sequence summary), `[SEP]` (segment boundary), `[MASK]` |
| Input embeddings | token + sin/cos position | token (+ learned position or RoPE) | token + **learned** position + **segment** (sentence A / B) |
| Norm | Post-LN | Pre-LN / RMSNorm | Post-LN in the paper (we use **Pre-LN**, like `vit.ipynb`) |
| Used for | seq2seq (translation) | text generation, chat | understanding: classification, NER, QA, retrieval embeddings |
| Can generate text? | yes (decoder) | yes, left to right | **no** (no causal factorization; only fills in blanks) |

Here: word-level tokens on the Enron spam emails, a ~6M-param BERT, MLM pretraining on the emails, then spam/ham fine-tuning from the `[CLS]` token, compared to the same model trained from scratch.

Shape flow: `ids, segments (B, T) -> token + position + segment embed (B, T, D) -> [LN -> MHA(pad mask) -> + -> LN -> GELU FFN -> +] x L -> LN -> (B, T, D) -> MLM logits at masked positions (M, V) | [CLS] (B, D) -> pooler -> logits (B, 2)`

In [ ]:
import re
import copy
import math
from collections import Counter

import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(42)

## 1. Data - Enron spam emails + word-level tokenizer
33,716 emails (`Subject`, `Message`, `Spam/Ham`), roughly half spam. 80/20 train/test split.

Each email becomes one BERT input with **two segments**: the subject is sentence A, the body is sentence B:

`[CLS] subject tokens [SEP] message tokens [SEP] [PAD] ...` - truncated/padded to $T = 128$.

Tokenizer: lowercase, regex split into words / numbers / single punctuation, keep the 10k most frequent training words, everything else -> `[UNK]`. (Real BERT uses a 30k **WordPiece** subword vocab, so it has almost no `[UNK]`.)

MLM pretraining only needs text, so it uses all training emails without labels. Fine-tuning uses just `n_labeled = 1000` labeled training emails - the typical setting where pretraining pays off (lots of raw text, few labels).

In [ ]:
df = pd.read_csv("../datasets/enron_spam_data.csv")
df["Subject"] = df["Subject"].fillna("")
df["Message"] = df["Message"].fillna("")
df["label"] = (df["Spam/Ham"] == "spam").astype(int)   # 1 = spam, 0 = ham
df = df.sample(frac=1.0, random_state=0).reset_index(drop=True)
n_train = int(0.8 * len(df))
train_df, test_df = df[:n_train], df[n_train:]

PAD, CLS, SEP, MASK, UNK = 0, 1, 2, 3, 4
specials = ["[PAD]", "[CLS]", "[SEP]", "[MASK]", "[UNK]"]
vocab_size = 10000
max_len = 128        # T
max_subject = 16     # subject tokens kept before the body

def tokenize(text):
    return re.findall(r"\[mask\]|[a-z0-9]+|[^\sa-z0-9]", text.lower()[:3000])  # 3000 chars is plenty for 128 tokens

counts = Counter(w for t in train_df["Subject"] + " " + train_df["Message"] for w in tokenize(t))
itos = specials + [w for w, _ in counts.most_common(vocab_size - len(specials))]
stoi = {w: i for i, w in enumerate(itos)}

def encode_words(text):
    return [MASK if w == "[mask]" else stoi.get(w, UNK) for w in tokenize(text)]

def encode(subject, message):
    # -> ids (T,), segments (T,): [CLS] A [SEP] B [SEP] [PAD]...
    a = encode_words(subject)[:max_subject]
    b = encode_words(message)[:max_len - len(a) - 3]
    ids = [CLS] + a + [SEP] + b + [SEP]
    seg = [0] * (len(a) + 2) + [1] * (len(b) + 1)
    pad = max_len - len(ids)
    return ids + [PAD] * pad, seg + [0] * pad

def encode_df(d):
    pairs = [encode(s, m) for s, m in zip(d["Subject"], d["Message"])]
    ids = torch.tensor([p[0] for p in pairs])     # (N, T)
    seg = torch.tensor([p[1] for p in pairs])     # (N, T)
    y = torch.tensor(d["label"].values)           # (N,)
    return ids, seg, y

train_ids, train_seg, train_y = encode_df(train_df)
test_ids, test_seg, test_y = encode_df(test_df)

print("vocab:", len(itos), "| train:", len(train_ids), "| test:", len(test_ids))
print("avg non-pad tokens:", (train_ids != PAD).sum(1).float().mean().item(), "| [UNK] rate:", ((train_ids == UNK).sum() / (train_ids != PAD).sum()).item())
print(" ".join(itos[i] for i in train_ids[0][:40].tolist()))

## 2. Input embeddings
BERT sums three learned embeddings per position: **which word**, **where** it is, and **which segment** (sentence A or B) it belongs to. Then LayerNorm + dropout (as in the paper).

### Formula: BERT input embedding
$$h_0^{(t)} = \text{Dropout}\big(\text{LN}(E_{tok}[x_t] + E_{pos}[t] + E_{seg}[s_t])\big), \qquad t = 1, \dots, T$$
- $x_t$: token id at position $t$ (vocab $V = 10{,}000$)
- $s_t \in \{0, 1\}$: segment id (0 = `[CLS]` + subject + first `[SEP]`, 1 = body + last `[SEP]`)
- $E_{tok} \in \mathbb{R}^{V \times D}$: token embedding (also reused as the MLM output layer, see §5)
- $E_{pos} \in \mathbb{R}^{T \times D}$: **learned** positional embedding, $T = 128$ (BERT: 512)
- $E_{seg} \in \mathbb{R}^{2 \times D}$: segment embedding
- $D = 256$: model dim (BERT-base: 768)
- $h_0 \in \mathbb{R}^{T \times D}$: input to the first encoder block

In [ ]:
class BertEmbeddings(nn.Module):
    # ids, segments (B, T) -> (B, T, D)
    def __init__(self, vocab_size, max_len, dim, dropout=0.1):
        super().__init__()
        self.tok = nn.Embedding(vocab_size, dim, padding_idx=PAD)
        self.pos = nn.Embedding(max_len, dim)
        self.seg = nn.Embedding(2, dim)
        self.ln = nn.LayerNorm(dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, ids, seg):
        pos = torch.arange(ids.shape[1], device=ids.device)       # (T,)
        x = self.tok(ids) + self.pos(pos) + self.seg(seg)         # (B, T, D) + (T, D) + (B, T, D)
        return self.dropout(self.ln(x))                            # (B, T, D)

## 3. Multi-head self-attention with a padding mask
Same scaled dot-product attention as the Transformer / ViT. BERT is an encoder, so there is **no causal mask**: every token attends to both left and right context - that is what "bidirectional" means.

The only mask is for **padding**: emails have different lengths, so `[PAD]` keys get $-\infty$ and receive zero attention weight. (Padded *queries* still produce outputs, but they are ignored by the losses.)

### Formula: Scaled dot-product attention with padding mask
$$\text{Attention}(Q,K,V) = \text{softmax}\!\left(\frac{QK^T}{\sqrt{d_k}} + M\right)V, \qquad M_{ij} = \begin{cases} 0 & x_j \ne \texttt{[PAD]} \\ -\infty & x_j = \texttt{[PAD]} \end{cases}$$
- $Q = XW_Q,\ K = XW_K,\ V = XW_V$: queries, keys, values, each $T \times d_k$ per head ($T = 128$)
- $d_k$: per-head dim ($D / h = 256 / 4 = 64$); $\sqrt{d_k} = 8$ keeps scores at unit variance
- $QK^T \in \mathbb{R}^{T \times T}$: score of query $i$ against key $j$
- $M$: depends only on the key $j$ (column), same for every query; `[CLS]` is never padding, so no row is all $-\infty$
- compare GPT: $M_{ij} = -\infty$ for $j > i$ (causal); BERT has no such term

### Formula: Multi-head attention
$$\text{MHA}(X) = \text{Concat}(\text{head}_1, \dots, \text{head}_h)\,W_O, \qquad \text{head}_i = \text{Attention}(XW_Q^i, XW_K^i, XW_V^i)$$
- $h = 4$ heads, each attends in its own 64-dim subspace
- $W_O \in \mathbb{R}^{D \times D}$: output projection (`self.proj`)

In [ ]:
class MultiHeadAttention(nn.Module):
    # (B, T, D), pad_mask (B, T) -> (B, T, D)
    def __init__(self, dim, num_heads, dropout=0.1):
        super().__init__()
        assert dim % num_heads == 0
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.qkv = nn.Linear(dim, 3 * dim)   # Q, K, V in one matmul
        self.proj = nn.Linear(dim, dim)      # W_O
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, pad_mask):
        B, T, D = x.shape
        qkv = self.qkv(x).reshape(B, T, 3, self.num_heads, self.head_dim)  # (B, T, 3D) -> (B, T, 3, heads, hd)
        q, k, v = qkv.permute(2, 0, 3, 1, 4)                                # each (B, heads, T, hd)

        att = (q @ k.transpose(-2, -1)) / self.head_dim ** 0.5              # (B, heads, T, T)
        att = att.masked_fill(pad_mask[:, None, None, :], float("-inf"))    # mask (B, 1, 1, T): hide [PAD] keys
        att = self.dropout(att.softmax(dim=-1))

        out = (att @ v).transpose(1, 2).reshape(B, T, D)                    # (B, heads, T, hd) -> concat heads (B, T, D)
        return self.proj(out)

## 4. Encoder block
Original BERT uses the Transformer's **Post-LN**: `x = LN(x + MHA(x))`, `x = LN(x + FFN(x))`. We use **Pre-LN** (like `vit.ipynb`): it keeps a clean residual path and trains stably without careful warmup, which matters for a small model trained briefly. The FFN uses **GELU** (BERT introduced it to Transformers; the 2017 model used ReLU).

### Formula: LayerNorm
$$\text{LN}(x) = \gamma \odot \frac{x - \mu}{\sqrt{\sigma^2 + \epsilon}} + \beta$$
- $x \in \mathbb{R}^{D}$: one token's features ($D = 256$); $\mu, \sigma^2$: their mean and variance
- $\gamma, \beta \in \mathbb{R}^D$: learned scale and shift; $\epsilon = 10^{-5}$

### Formula: Pre-LN encoder block
$$h' = h + \text{MHA}(\text{LN}(h)), \qquad h_{out} = h' + \text{FFN}(\text{LN}(h'))$$
- $h \in \mathbb{R}^{T \times D}$: block input; the stack has $L = 4$ blocks (BERT-base: 12)
- Post-LN (paper): $h' = \text{LN}(h + \text{MHA}(h))$, $h_{out} = \text{LN}(h' + \text{FFN}(h'))$

### Formula: GELU feed-forward
$$\text{FFN}(x) = W_2\,\text{GELU}(W_1 x + b_1) + b_2, \qquad \text{GELU}(u) = u\,\Phi(u)$$
- $W_1 \in \mathbb{R}^{4D \times D}$ (256 -> 1024), $W_2 \in \mathbb{R}^{D \times 4D}$ (1024 -> 256)
- $\Phi$: standard normal CDF; GELU is a smooth ReLU that lets small negative values through

In [ ]:
class EncoderBlock(nn.Module):
    # (B, T, D) -> (B, T, D)
    def __init__(self, dim, num_heads, mlp_ratio=4, dropout=0.1):
        super().__init__()
        self.ln1 = nn.LayerNorm(dim)
        self.attn = MultiHeadAttention(dim, num_heads, dropout)
        self.ln2 = nn.LayerNorm(dim)
        self.ffn = nn.Sequential(
            nn.Linear(dim, mlp_ratio * dim),
            nn.GELU(),
            nn.Linear(mlp_ratio * dim, dim),
            nn.Dropout(dropout),
        )
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, pad_mask):
        x = x + self.dropout(self.attn(self.ln1(x), pad_mask))  # Pre-LN, (B, T, D)
        x = x + self.ffn(self.ln2(x))                           # (B, T, D) -> (B, T, 4D) -> (B, T, D)
        return x

## 5. BERT model + heads
- **Encoder**: embeddings -> $L$ blocks -> final LN (needed with Pre-LN).
- **MLM head** (pretraining): `Linear -> GELU -> LN`, then project onto the vocab with the **token embedding matrix** (weight tying) plus a per-word bias. It is applied only at the masked positions, which saves computing $B \times T \times V$ logits.
- **Classification head** (fine-tuning): BERT's **pooler** ($\tanh$ layer on the `[CLS]` output) followed by a linear classifier. `[CLS]` has no word of its own, so through training it learns to summarize the whole sequence.

### Formula: MLM head with weight tying
$$z_t = E_{tok}\,\text{LN}\big(\text{GELU}(W_m h_L^{(t)} + b_m)\big) + b_V$$
- $h_L^{(t)} \in \mathbb{R}^D$: final hidden state at a masked position $t$
- $W_m \in \mathbb{R}^{D \times D}$: transform layer
- $E_{tok} \in \mathbb{R}^{V \times D}$: the input token embedding, reused as output weights (saves $V D = 2.56$M params)
- $b_V \in \mathbb{R}^V$: output bias; $z_t \in \mathbb{R}^{V}$: logits over the 10k words

### Formula: [CLS] classification head
$$p(y \mid x) = \text{softmax}\big(W_c \tanh(W_p h_L^{(\text{CLS})} + b_p) + b_c\big)$$
- $h_L^{(\text{CLS})}$: final hidden state of position 0
- $W_p \in \mathbb{R}^{D \times D}$: pooler (in real BERT it is pretrained with NSP; here it starts random)
- $W_c \in \mathbb{R}^{2 \times D}$: classifier; $y \in \{\text{ham}, \text{spam}\}$

In [ ]:
class BERT(nn.Module):
    # ids, seg (B, T) -> hidden (B, T, D); heads: mlm_logits, classify
    def __init__(self, vocab_size, max_len, dim=256, depth=4, num_heads=4, num_classes=2, dropout=0.1):
        super().__init__()
        self.embed = BertEmbeddings(vocab_size, max_len, dim, dropout)
        self.blocks = nn.ModuleList([EncoderBlock(dim, num_heads, dropout=dropout) for _ in range(depth)])
        self.ln = nn.LayerNorm(dim)

        # MLM head
        self.mlm_transform = nn.Sequential(nn.Linear(dim, dim), nn.GELU(), nn.LayerNorm(dim))
        self.mlm_bias = nn.Parameter(torch.zeros(vocab_size))

        # classification head on [CLS]
        self.pooler = nn.Sequential(nn.Linear(dim, dim), nn.Tanh())
        self.cls_dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(dim, num_classes)

        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, std=0.02)   # BERT init: N(0, 0.02)
        if isinstance(m, nn.Linear) and m.bias is not None:
            nn.init.zeros_(m.bias)

    def forward(self, ids, seg):
        pad_mask = ids == PAD                     # (B, T), True = padding
        x = self.embed(ids, seg)                  # (B, T, D)
        for block in self.blocks:
            x = block(x, pad_mask)                # (B, T, D)
        return self.ln(x)                         # (B, T, D)

    def mlm_logits(self, h):
        # h (M, D) hidden states at masked positions -> (M, V)
        return self.mlm_transform(h) @ self.embed.tok.weight.T + self.mlm_bias  # weight tying: (M, D) @ (D, V)

    def classify(self, ids, seg):
        h = self.forward(ids, seg)[:, 0]          # [CLS]: (B, T, D) -> (B, D)
        return self.classifier(self.cls_dropout(self.pooler(h)))  # (B, D) -> (B, 2)

model = BERT(vocab_size, max_len).to(device)
print(f"params: {sum(p.numel() for p in model.parameters()) / 1e6:.2f}M")
print(model(train_ids[:2].to(device), train_seg[:2].to(device)).shape, model.classify(train_ids[:2].to(device), train_seg[:2].to(device)).shape)

## 6. MLM masking (15%, 80/10/10)
Pick 15% of the real word positions (never `[PAD]`, `[CLS]`, `[SEP]`) as prediction targets. Of those:
- 80% -> replaced by `[MASK]`
- 10% -> replaced by a random word
- 10% -> left unchanged

Why not always `[MASK]`? `[MASK]` never appears at fine-tuning time. The random/unchanged cases force the model to keep a good representation of **every** token, since any token might be a target, and it can't just trust the input word.

### Formula: Masking rule
$$m_t \sim \text{Bernoulli}(0.15)\ \ (t \text{ a word position}), \qquad \tilde x_t = \begin{cases} \texttt{[MASK]} & \text{w.p. } 0.8 \\ w \sim \text{Uniform(words)} & \text{w.p. } 0.1 \\ x_t & \text{w.p. } 0.1 \end{cases} \quad \text{if } m_t = 1, \qquad \tilde x_t = x_t \ \text{ otherwise}$$
- $x_t$: original token; $\tilde x_t$: token fed to the model
- $m_t$: 1 if position $t$ is a prediction target (label = $x_t$), else label = $-100$ (ignored by the loss)
- per email ~0.15 x ~120 words = ~18 targets

In [ ]:
def mask_tokens(ids, p=0.15):
    # ids (B, T) -> masked input (B, T), labels (B, T) with -100 at non-target positions
    ids = ids.clone()
    labels = ids.clone()
    can_mask = ids > UNK                                           # real words only ([UNK] is not a useful target either)
    target = (torch.rand(ids.shape, device=ids.device) < p) & can_mask  # (B, T) 15% of words
    labels[~target] = -100

    r = torch.rand(ids.shape, device=ids.device)
    ids[target & (r < 0.8)] = MASK                                 # 80% -> [MASK]
    rand_pos = target & (r >= 0.8) & (r < 0.9)                     # 10% -> random word
    ids[rand_pos] = torch.randint(len(specials), vocab_size, (int(rand_pos.sum()),), device=ids.device)
    return ids, labels                                             # remaining 10% keep the original word

x, lab = mask_tokens(train_ids[:1])
print("input: ", " ".join(itos[i] for i in x[0][:30].tolist()))
print("target:", " ".join(itos[i] if i != -100 else "_" for i in lab[0][:30].tolist()))

## 7. MLM pretraining
Loss is cross-entropy **only over the masked positions** - the other 85% of tokens give no gradient signal directly (one reason MLM is less sample-efficient per token than next-token prediction, which predicts every position). We also track masked-token accuracy.

BERT's optimizer recipe: **AdamW** (weight decay 0.01), **linear warmup then linear decay** to 0.

### Formula: MLM loss
$$\mathcal{L}_{MLM} = -\frac{1}{|\mathcal{M}|}\sum_{(b,t) \in \mathcal{M}} \log p_\theta\big(x_{b,t} \mid \tilde x_b\big)$$
- $\mathcal{M}$: set of target positions in the batch (label $\ne -100$), $|\mathcal{M}| \approx 0.15 \cdot B \cdot$ (words per email) with $B = 64$
- $\tilde x_b$: the corrupted email $b$ (the model sees both sides of $t$)
- $p_\theta(\cdot \mid \tilde x_b) = \text{softmax}(z_t)$ over $V = 10{,}000$ words; uniform guessing gives $\log 10000 \approx 9.2$

### Formula: AdamW
$$\theta_t = \theta_{t-1} - \eta_t\left(\frac{\hat m_t}{\sqrt{\hat v_t} + \epsilon} + \lambda\,\theta_{t-1}\right)$$
- $\hat m_t, \hat v_t$: bias-corrected EMAs of the gradient and squared gradient ($\beta_1 = 0.9, \beta_2 = 0.999$)
- $\eta_t$: learning rate from the schedule below (peak $5 \times 10^{-4}$); $\epsilon = 10^{-8}$
- $\lambda = 0.01$: decoupled weight decay (applied to $\theta$ directly, not through the gradient)

### Formula: Linear warmup + linear decay
$$\eta_t = \eta \cdot \min\!\left(\frac{t}{T_w},\ \frac{T_{total} - t}{T_{total} - T_w}\right)$$
- $\eta$: peak LR ($5 \times 10^{-4}$); $t$: optimizer step
- $T_w$: warmup steps (300); $T_{total}$: total steps (3000)
- ramps up while Adam's moment estimates are noisy, then decays linearly to 0 at the end

In [ ]:
pretrain_steps = 3000
batch_size = 64
lr = 5e-4
warmup = 300
log_every = 250

def warmup_linear(warmup, total):
    return lambda t: min((t + 1) / warmup, max(0.0, (total - t) / (total - warmup)))

optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, warmup_linear(warmup, pretrain_steps))

def mlm_step(model, ids, seg):
    # -> loss, number correct, number of targets
    x, labels = mask_tokens(ids)                           # (B, T), (B, T)
    h = model(x, seg)                                      # (B, T, D)
    target = labels != -100                                # (B, T)
    logits = model.mlm_logits(h[target])                   # (M, D) -> (M, V), M = #targets
    loss = F.cross_entropy(logits, labels[target])         # (M, V) vs (M,) -> scalar
    correct = (logits.argmax(-1) == labels[target]).sum().item()
    return loss, correct, target.sum().item()

@torch.no_grad()
def mlm_eval(model, n_batches=20):
    model.eval()
    g = torch.Generator().manual_seed(0)
    tot_loss = tot_correct = tot_n = 0
    for _ in range(n_batches):
        ix = torch.randint(len(test_ids), (batch_size,), generator=g)
        loss, c, n = mlm_step(model, test_ids[ix].to(device), test_seg[ix].to(device))
        tot_loss += loss.item() * n; tot_correct += c; tot_n += n
    model.train()
    return tot_loss / tot_n, tot_correct / tot_n

history = []
model.train()
run_loss = run_correct = run_n = 0
for step in range(1, pretrain_steps + 1):
    ix = torch.randint(len(train_ids), (batch_size,))
    loss, c, n = mlm_step(model, train_ids[ix].to(device), train_seg[ix].to(device))

    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    scheduler.step()
    run_loss += loss.item(); run_correct += c; run_n += n

    if step % log_every == 0:
        test_loss, test_acc = mlm_eval(model)
        history.append((step, run_loss / log_every, test_loss))
        print(f"step {step:5d} | train mlm loss {run_loss / log_every:.3f} | masked acc {run_correct / run_n:.3f} | test mlm loss {test_loss:.3f} | test masked acc {test_acc:.3f}")
        run_loss = run_correct = run_n = 0

steps, tr, te = zip(*history)
plt.plot(steps, tr, label="train")
plt.plot(steps, te, label="test")
plt.xlabel("step"); plt.ylabel("MLM loss"); plt.legend(); plt.show()

## 8. Fill in the [MASK]
The pretrained encoder can predict a hidden word from context on both sides. This is all BERT can "generate": it fills blanks in place, it cannot write text left to right.

In [ ]:
@torch.no_grad()
def fill_mask(model, subject, message, k=5):
    model.eval()
    ids, seg = encode(subject, message)
    ids, seg = torch.tensor([ids], device=device), torch.tensor([seg], device=device)  # (1, T)
    h = model(ids, seg)[0]                                  # (T, D)
    pos = (ids[0] == MASK).nonzero().flatten()              # (n_masks,)
    top = model.mlm_logits(h[pos]).topk(k, dim=-1).indices  # (n_masks, V) -> (n_masks, k)
    print(f"{subject} | {message}")
    for p, t in zip(pos.tolist(), top.tolist()):
        print(f"   pos {p:3d}: {[itos[i] for i in t]}")

fill_mask(model, "meeting tomorrow", "please [MASK] me if you have any questions .")
fill_mask(model, "re : gas nomination", "the [MASK] for the pipeline will be sent to you by friday .")
fill_mask(model, "cheap meds", "buy viagra online at the lowest [MASK] ! click here to [MASK] .")
fill_mask(model, "fw : report", "thanks , [MASK] . i will forward the attached [MASK] to the team .")

## 9. Fine-tuning: spam vs ham from [CLS]
Take the pretrained encoder, add nothing but the (already present) pooler + classifier, and train **all** weights on `n_labeled = 1000` labeled emails with a smaller LR. The baseline is the **same architecture from random init**, trained with the exact same data, steps and schedule - the gap is what pretraining bought.

### Formula: Classification cross-entropy
$$\mathcal{L}_{CLS} = -\frac{1}{B}\sum_{b=1}^{B} \log p_\theta\big(y_b \mid x_b\big)$$
- $B = 32$: batch size; $y_b \in \{0 = \text{ham}, 1 = \text{spam}\}$
- $p_\theta(y \mid x)$: softmax of the `[CLS]` head logits (§5)
- uniform guess gives $\log 2 \approx 0.69$
- same AdamW + warmup/linear decay as pretraining, peak LR $10^{-4}$, $T_w = 50$, $T_{total} = 600$

In [ ]:
n_labeled = 1000
ft_steps = 600
ft_batch = 32
ft_lr = 1e-4
ft_warmup = 50
eval_every = 100

g = torch.Generator().manual_seed(0)
lab_ix = torch.randperm(len(train_ids), generator=g)[:n_labeled]
ft_ids, ft_seg, ft_y = train_ids[lab_ix], train_seg[lab_ix], train_y[lab_ix]
print("labeled set spam fraction:", ft_y.float().mean().item())

@torch.no_grad()
def cls_accuracy(model, ids, seg, y, bs=256):
    model.eval()
    correct = 0
    for i in range(0, len(ids), bs):
        logits = model.classify(ids[i:i + bs].to(device), seg[i:i + bs].to(device))  # (bs, 2)
        correct += (logits.argmax(-1).cpu() == y[i:i + bs]).sum().item()
    model.train()
    return correct / len(ids)

def finetune(model, name):
    opt = torch.optim.AdamW(model.parameters(), lr=ft_lr, weight_decay=0.01)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, warmup_linear(ft_warmup, ft_steps))
    torch.manual_seed(0)                                     # same batches for both runs
    curve = []
    model.train()
    for step in range(1, ft_steps + 1):
        ix = torch.randint(n_labeled, (ft_batch,))
        logits = model.classify(ft_ids[ix].to(device), ft_seg[ix].to(device))  # (B, 2)
        loss = F.cross_entropy(logits, ft_y[ix].to(device))                   # (B, 2) vs (B,) -> scalar

        opt.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        sched.step()

        if step % eval_every == 0:
            acc = cls_accuracy(model, test_ids, test_seg, test_y)
            curve.append((step, acc))
            print(f"[{name}] step {step:4d} | loss {loss.item():.4f} | test acc {acc:.4f}")
    return curve

pretrained = copy.deepcopy(model)                             # keep the MLM model intact
curve_pt = finetune(pretrained, "pretrained")

torch.manual_seed(42)
scratch = BERT(vocab_size, max_len).to(device)                # same architecture, random init
curve_sc = finetune(scratch, "scratch")

## 10. Pretrained vs from scratch

In [ ]:
print(f"test accuracy after {ft_steps} steps on {n_labeled} labels:")
print(f"  pretrained (MLM) : {curve_pt[-1][1]:.4f}")
print(f"  from scratch     : {curve_sc[-1][1]:.4f}")

plt.plot(*zip(*curve_pt), marker="o", label="MLM-pretrained")
plt.plot(*zip(*curve_sc), marker="o", label="random init")
plt.xlabel("fine-tuning step"); plt.ylabel("test accuracy"); plt.legend(); plt.show()

## Notes
- Pretraining should give higher test accuracy and faster convergence with only 1000 labels; the from-scratch model has to learn word meanings from the labels alone and overfits. With all ~27k labels the gap shrinks: spam detection is easy (even bag-of-words models reach ~98%), which is why we fine-tune on a small subset.
- Here pretraining and fine-tuning use the same emails (in-domain). Real BERT pretrains once on BooksCorpus + Wikipedia (3.3B words, 1M steps of 256 x 512 tokens) and fine-tunes on many tasks; BERT-base is 110M params ($L=12, D=768, h=12$), BERT-large 340M.
- Skipped from the paper: **next-sentence prediction** (a binary `[CLS]` task: is segment B the real continuation of A?). RoBERTa (Liu et al., 2019) showed removing NSP doesn't hurt and that training longer on more data with dynamic masking (re-masking every batch, as we do) helps more. We still use two segments (subject / body) so the segment embeddings are exercised.
- Descendants: RoBERTa (better recipe), ALBERT (shared layers, factorized embeddings), DistilBERT (distillation), ELECTRA (replaced-token detection, learns from all tokens instead of 15%), DeBERTa (disentangled relative position). Sentence-embedding and reranker models used in retrieval/RAG are still mostly BERT-style encoders.
- Word-level vocab is a simplification: 10k words leaves some `[UNK]`s; WordPiece/BPE splits rare words into known subwords instead.